In [1]:
import sys
import os
import logging
import pandas as pd
import numpy as np

#Configure the logging to append to pipeline.log
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler("pipeline.log", mode='a', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger("Build a Traffic Recommendation System")
data_source = "ml_traffic_accident_proxy.csv"

logger.info("========================================================")
logger.info("LAUNCHING TASK 5: BUILD A TRAFFIC RECOMMENDATION SYSTEM")
logger.info("========================================================")

if not os.path.exists(data_source):
    logger.error(f"Inbound Data Source Error: {data_source} missing. Execute Proxy_accident-risk_label.py first.")
    sys.exit(1)

df = pd.read_csv(data_source)
logger.info("Recommendation engine operational cache connected successfully.")

#=====================================================================
#2. CORE RECOMMENDATION ENGINE FUNCTION
#=====================================================================
def generate_traffic_recommendation(is_weekend_input: int, weather_condition_input: str) -> str:
    """
    Core Recommendation System: Automatically extracts optimized daylight windows 
    based on historical carrying capacities, day types, and active climate hazards.
    """
    clean_weather = weather_condition_input.strip().capitalize()
    day_label = "Weekend" if is_weekend_input == 1 else "Weekday"
    
    logger.info(f"Recommendation Engine Triggered | Constraints: Day_Type={day_label}, Weather={clean_weather}")
    
    #Filter master cache according to targeted Day Type constraints
    filtered_df = df[df['Is_Weekend'] == is_weekend_input]
    
    #Check for active one-hot encoded climate features
    target_weather_col = f"Weather_{clean_weather}"
    if target_weather_col in filtered_df.columns:
        weather_subset = filtered_df[filtered_df[target_weather_col] == 1]
        if len(weather_subset) > 100:
            filtered_df = weather_subset
            logger.info(f"Climate constraint matched: Isolate hours matching '{clean_weather}' matrix.")
        else:
            logger.warning(f"Thin data sample size for '{clean_weather}' on a {day_label}. Bypassing climate constraint slice to avoid skew.")
    else:
        logger.warning(f"Weather category '{clean_weather}' not recognized in schema. Defaulting to macro day-type distribution.")

    #Focus optimization search within daylight operational windows (06:00 to 20:00)
    daylight_df = filtered_df[(filtered_df['Hour'] >= 6) & (filtered_df['Hour'] <= 20)]
    
    #Group by hour and calculate the mean traffic volume to isolate low-traffic periods
    hourly_means = daylight_df.groupby('Hour')['traffic_volume'].mean()
    
    if hourly_means.empty:
        best_hour = 10
        mean_volume = 3200
    else:
        best_hour = hourly_means.idxmin()
        mean_volume = hourly_means.min()
        
    start_time_str = f"{best_hour:02d}:00"
    end_time_str = f"{(best_hour + 1):02d}:00"
    
    #Extract active hazard indicators to output proactive safety warnings
    is_freezing_hazard = "Weather_Snow" in filtered_df.columns and clean_weather in ['Snow', 'Rain', 'Thunderstorm', 'Squall']
    
    #Generate the structural Plain-Language Recommendation string output
    recommendation_text = (
        f"🎯 [SMART MOBILITY RECOMMENDATION] For a {day_label.lower()} journey under '{clean_weather.lower()}' atmospheric conditions, "
        f"the optimized departure window is between {start_time_str} and {end_time_str}. At this time, historical data demonstrates "
        f"that traffic volume reaches an optimized low-density mode averaging approximately {mean_volume:.0f} vehicles/hour. "
    )
    
    #FIXED RANGE BOUNDARY: Explicit comparison completely avoids the syntax loop bug
    if is_freezing_hazard:
        recommendation_text += "⚠️ WARNING: Active sub-zero winter precipitation is detected; high risk of black-ice formation. Reduce lane speeds."
    elif (7 <= best_hour <= 8) or (16 <= best_hour <= 17):
        recommendation_text += "⚠️ NOTICE: This boundary overlaps with regional peak commuting intervals. Expect minor queuing delays."
    else:
        recommendation_text += "✨ Lane flow conditions are optimized for maximum corridor transit throughput."

    logger.info("Recommendation output text compiled safely.")
    return recommendation_text


#=====================================================================
#3. SIMULATED APPLICATION EXECUTION RUNS
#=====================================================================
print("\n" + "="*80)
#Test Case 1: Standard Weekday Commute in Rainy Conditions
rec1 = generate_traffic_recommendation(is_weekend_input=0, weather_condition_input="Rain")
print(rec1)

print("\n" + "-"*80)
#Test Case 2: Weekend Journey in Clear Conditions
rec2 = generate_traffic_recommendation(is_weekend_input=1, weather_condition_input="Clear")
print(rec2)
print("="*80 + "\n")

logger.info("Task 5 Build a Traffic Recommendation System Engine pipeline executed successfully")
logger.info("====================================================")

2026-11-09 18:47:04 [INFO] Build a Traffic Recommendation System: ========================================================
2026-11-09 18:47:04 [INFO] Build a Traffic Recommendation System: LAUNCHING TASK 5: BUILD A TRAFFIC RECOMMENDATION SYSTEM
2026-11-09 18:47:04 [INFO] Build a Traffic Recommendation System: ========================================================
2026-11-09 18:47:04 [INFO] Build a Traffic Recommendation System: Recommendation engine operational cache connected successfully.

2026-11-09 18:47:04 [INFO] Build a Traffic Recommendation System: Recommendation Engine Triggered | Constraints: Day_Type=Weekday, Weather=Rain
2026-11-09 18:47:04 [INFO] Build a Traffic Recommendation System: Climate constraint matched: Isolate hours matching 'Rain' matrix.
2026-11-09 18:47:04 [INFO] Build a Traffic Recommendation System: Recommendation output text compiled safely.
🎯 [SMART MOBILITY RECOMMENDATION] For a weekday journey under 'rain' atmospheric conditions, the optimized departur